# Project Overview
This section aims to integrate, clean, and process macroeconomic and exchange rate data, focusing on the target index (US Dollar Index - DXY) and integrating it with the currency pair (USDCAD) in addition to 28 other economic indicators (such as interest rates, GDP, inflation, etc.) in the period between `1994-01-01` and `2021-01-10` to prepare it for training machine learning models.

In [ ]:
# Environment Setup and Imports
import os
import glob
import pandas as pd
import numpy as np
import warnings

# Hide warning messages to keep code output clean
warnings.filterwarnings('ignore')

# Defining the target time range for the study
START_DATE = '1994-01-01'
END_DATE = '2021-01-10'

In [ ]:
# Mounting Google Drive
from google.colab import drive
drive.mount('/content/drive')

# The main path to the folder containing all 28 index files
BASE_PATH = '/content/drive/MyDrive/ML_Project_Data'

Mounted at /content/drive


In [ ]:
# Data Loading & Primary Cleaning

# Primary file paths
# Correcting file paths as the files are located directly in /content/
dxy_path = '/content/US Dollar Index (DXY).csv'
usdcad_path = '/content/USDCAD.csv'

# 1. Processing the US Dollar Index (DXY) file
print("Processing the DXY file")
dxy_df = pd.read_csv(dxy_path)
dxy_df = dxy_df.dropna(how='all')
dxy_df['Date'] = pd.to_datetime(dxy_df['Date'])
# Filter data by time range
dxy_df = dxy_df[(dxy_df['Date'] >= START_DATE) & (dxy_df['Date'] <= END_DATE)]
dxy_df_cleaned = dxy_df.dropna().drop(columns=['Volume'], errors='ignore')

# 2. Processing the USDCAD file
print("Processing the USDCAD file")
usdcad_df = pd.read_csv(usdcad_path)
usdcad_df = usdcad_df.rename(columns={'time': 'Date'})
usdcad_df['Date'] = pd.to_datetime(usdcad_df['Date'])
# Filter data by time range
usdcad_df = usdcad_df[(usdcad_df['Date'] >= START_DATE) & (usdcad_df['Date'] <= END_DATE)]

# Synchronize USDCAD dates with dates available in DXY
dxy_valid_dates = dxy_df_cleaned['Date'].unique()
usdcad_df_filtered = usdcad_df[usdcad_df['Date'].isin(dxy_valid_dates)]

# Merge the two main files into a single data frame (DataFrame)
final_combined_df = pd.merge(dxy_df_cleaned, usdcad_df_filtered, on='Date', how='left')
print(f"Data after merging DXY and USDCAD: {final_combined_df.shape}")

Processing the DXY file
Processing the USDCAD file
Data after merging DXY and USDCAD: (6879, 13)


In [ ]:
# 5. Merging All Macroeconomic Features

print("All economic files are being processed")

all_csv_files = glob.glob(os.path.join(BASE_PATH, '*.csv'))

# Exclude merged files to avoid duplication
exclude_files = ['US Dollar Index (DXY).csv', 'USDCAD.csv']

for file_path in all_csv_files:
    file_name = os.path.basename(file_path)

# Skip Primary files
    if file_name in exclude_files:
        continue

    try:
# Load the file and skip any corrupted lines
        df_temp = pd.read_csv(file_path, on_bad_lines='skip')
        df_temp = df_temp.dropna(how='all')

# Automatic search for the date column (whether it is written as Date, DATE, date or time)
        date_col = next((c for c in df_temp.columns if c.strip().lower() in ['date', 'time']), None)

        if date_col:
# Unify the column name to 'Date' and convert it to DateTime format
            df_temp[date_col] = pd.to_datetime(df_temp[date_col], errors='coerce')
            df_temp = df_temp.rename(columns={date_col: 'Date'})

# Filtering the time range and removing missing values ​​from the temporary file
            mask = (df_temp['Date'] >= START_DATE) & (df_temp['Date'] <= END_DATE)
            df_temp = df_temp.loc[mask].dropna().copy()

# Merging the current index with the main dataset based on history
            final_combined_df = pd.merge(final_combined_df, df_temp, on='Date', how='left')
            print(f"The processing and merging were successful:{file_name}")
        else:
            print(f"Warning: No date column was found in the file {file_name}")

    except Exception as e:
        print(f"Error during file processing:{file_name}: {e}")

# Chronological ordering of data to ensure the accuracy of time series
final_combined_df = final_combined_df.sort_values('Date')

# Data Imputation
# Since economic data (such as GDP) is published monthly or quarterly,
# we use the ffill (Forward Fill) function to fill in the days on which there is no publication with the last value issued
final_combined_df = final_combined_df.ffill()

# Fill in any remaining values ​​at the beginning of the time series
final_combined_df = final_combined_df.bfill()

print(f"Final dimensions of the comprehensive dataset: {final_combined_df.shape}")

All economic files are being processed
Final dimensions of the comprehensive dataset: (6879, 13)


In [ ]:
# Final Results and Saving

# Show the first 5 rows to ensure the process is successful
print("Quick look at the data after the final merger:")
display(final_combined_df.head())

# Check for any missing values
missing_values = final_combined_df.isnull().sum().sum()
print(f"Total missing values ​​in the final data: {missing_values}")

# Save the final file ready for machine learning algorithms
output_path = '/content/final_merged_df.csv'
final_combined_df.to_csv(output_path, index=False)

print(f"All files were merged and the final data was saved successfully: {output_path}")

Quick look at the data after the final merger:


,Date,Open,High,Low,Close,Adj Close,open,high,low,close,tick_volume,spread,real_volume
0,1994-01-03,96.760002,97.019997,96.540001,96.970001,96.970001,1.3270,1.3300,1.3120,1.3141,2311.0,50.0,0.0
1,1994-01-04,96.980003,97.089996,96.459999,96.550003,96.550003,1.3139,1.3185,1.3125,1.3169,901.0,50.0,0.0
2,1994-01-05,96.510002,96.720001,96.389999,96.580002,96.580002,1.3167,1.3226,1.3105,1.3212,1971.0,50.0,0.0
3,1994-01-06,96.769997,96.889999,96.300003,96.820000,96.820000,1.3205,1.3240,1.3165,1.3195,1401.0,50.0,0.0
4,1994-01-07,96.750000,96.849998,95.989998,96.080002,96.080002,1.3190,1.3248,1.3175,1.3195,1411.0,50.0,0.0


Total missing values ​​in the final data: 0
All files were merged and the final data was saved successfully: /content/final_merged_df.csv
